# 🧪 Thermal paper — fix-and-verify experiments (single Kaggle notebook)

This notebook re-runs **every experiment the paper's claims depend on** on **one fixed split**, with **several seeds** (mean ± std), and writes ready-to-paste LaTeX tables and figures. It is self-contained — it does not need your other notebooks.

| § | What runs | What it fixes in the paper |
|---|---|---|
| 3 | **Leakage check** (seconds, no training) | tells you whether the same frame (other palette) or a near-identical frame sits in both train and test |
| 9 | **Clean ablation**: Baseline · +Aug only · +GRL only · +PaIN only · +Ortho only · +Focal only · +GRL+PaIN · +GRL+PaIN+Ortho · Full Model | Table VI (reviewer comment 4), the "Full Model" row, Table V "Ours" |
| 9 | **Full Model + SimCLR init** (pre-trained on TRAIN images only) | whether N7 can stay as a contribution |
| 9 | **Full Model + CLUB** | CLUB on the same split as N2 |
| 9 | **Identity loss: CE vs ArcFace vs CosFace vs CE+SupCon**, all evaluated correctly | Table VII and the ArcFace claim |
| 10 | **Cross-palette accuracy on the TEST split only** | Table IV |
| 10 | **Mechanism probes** (how much palette / cross-task information the features carry) | evidence for N1, N2, N8, CLUB |
| 11 | **Real task-specific Grad-CAM** + attention share per face region | Fig. 5 and the XAI claim |

**Two bugs from the original notebook are fixed here:** (1) the orthogonality loss (N2) now applies to every image — before, it only applied to images with *both* labels, and none exist; (2) ArcFace/CosFace now predict with the same cosine classifier they are trained with — before, accuracy was read from a head that never trained, which is what produced 2.43%.

### How to run on Kaggle
1. **File → Import Notebook** → upload this file. **Add Data** → your *Comprehensive Facial Thermal Dataset*.
2. **Settings → Accelerator: GPU T4** (P100 also fine) · **Internet: On** (downloads ImageNet weights).
3. In the first code cell set `SMOKE_TEST = True` and **Run All** (~10 min). This checks every cell on your data. **Read the leakage verdict printed in §3.**
4. Set `SMOKE_TEST = False` → **Save Version → Save & Run All (Commit)**. It runs in the background; you can close the browser.
5. When it finishes: open the version → **Output** → download `paper_fixes_<split>.zip`. Send me `summary.md` (it contains every table) and I will update the paper.

**Time:** about 2 h per seed (≈ 6 h for `SEEDS = [42, 43, 44]`). The notebook prints a real estimate after the first run. If a session dies, re-running the notebook skips runs that already finished (as long as the output folder still exists).

In [ ]:
# ═══════════════════════════ SETTINGS — the only cell you need to edit ═══════════════════════════
SMOKE_TEST  = False            # True → 1 epoch, 1 seed, small subset (~10 min): checks the whole pipeline on your data
SEEDS       = [42, 43, 44]     # training seeds → tables report mean ± std.  [42] alone takes ~1/3 of the time
EPOCHS      = 25               # same as the 7-model comparison in the paper
SPLIT_MODE  = "random"         # "random"  = the paper's protocol (stratified by class × palette)
                               # "frame"   = all palette renderings of the same frame stay in the same split
                               # "session" = whole recording sessions stay in the same split (strictest)
RUN_SIMCLR  = True             # N7: SimCLR pre-training on TRAIN images only, then fine-tune the Full Model
RUN_CLUB    = True             # Full Model + CLUB mutual-information penalty
RUN_LOSSES  = True             # ArcFace / CosFace / CE+SupCon with correct evaluation
RUN_PROBES  = True             # linear probes on the learned features (mechanism evidence)
RUN_GRADCAM = True             # real Grad-CAM on the Full Model + attention share per face region

## 1 · Install and imports

In [ ]:
import os, sys, subprocess
# albumentations >= 2.0 is needed for the augmentation API below (same as your earlier Kaggle runs)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "albumentations"], check=False)

import re, json, math, time, random, shutil, bisect, traceback, warnings
from collections import defaultdict
from contextlib import nullcontext
from datetime import datetime

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.ticker import MaxNLocator
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
from torch.optim.lr_scheduler import CosineAnnealingLR
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
print(f"✓ torch {torch.__version__} | albumentations {A.__version__} | CUDA available: {torch.cuda.is_available()}")

## 2 · Configuration and dataset scan

In [ ]:
def find_dataset_root():
    """Find the folder that contains 'Facial Emotion' and 'Face Recognetion'."""
    markers = {"Facial Emotion", "Facial emotion", "facial emotion",
               "Face Recognetion", "Face Recognition", "face recognition", "face recognetion"}
    for root in ["/kaggle/input", "/content/drive/MyDrive", "./", "../"]:
        if not os.path.isdir(root):
            continue
        for cur, dirs, _ in os.walk(root):
            if cur[len(root):].count(os.sep) > 5:
                dirs.clear(); continue
            if markers.intersection(dirs):
                return cur
    raise FileNotFoundError("Dataset not found. Attach it with 'Add Data' and check that it contains "
                            "the 'Facial Emotion' and 'Face Recognetion' folders.")

def find_subdir(parent, names):
    for n in names:
        if os.path.isdir(os.path.join(parent, n)):
            return os.path.join(parent, n)
    raise FileNotFoundError(f"None of {names} found inside {parent}")

DATASET_ROOT  = find_dataset_root()
EMOTION_ROOT  = find_subdir(DATASET_ROOT, ["Facial Emotion", "Facial emotion", "facial emotion"])
IDENTITY_ROOT = find_subdir(DATASET_ROOT, ["Face Recognetion", "Face Recognition", "face recognition", "face recognetion"])

EMOTIONS    = ["angry", "happy", "natural", "sad", "surprise"]
EMO_ALIASES = {"surpise": "surprise"}          # the dataset folder name is misspelled
PALETTES    = ["ICEBLUE", "IRONBOW", "RAIN", "Red_Hot", "White_Hot"]
IMG_EXTS    = (".bmp", ".png", ".jpg", ".jpeg")

# ── hyper-parameters (same values as the paper) ──────────────────────────────
IMG_SIZE, BATCH_SIZE = 224, 16                 # batch = 8 emotion + 8 identity images
LR, WEIGHT_DECAY     = 3e-4, 1e-4
GRL_ALPHA_MAX        = 0.5
W_EMO, W_ID, W_PAL, W_ORT = 2.0, 1.0, 0.05, 0.01
W_CLUB               = 0.1
W_SUPCON, SUPCON_T   = 0.5, 0.1
ARC_S, ARC_M         = 30.0, 0.30
COS_S, COS_M         = 30.0, 0.35
SIMCLR_EPOCHS, SIMCLR_BATCH, SIMCLR_TAU = 20, 32, 0.5
SPLIT_SEED           = 42
VAL_R, TEST_R        = 0.15, 0.10              # 75 / 15 / 10
GAP_SECONDS          = 30    # frames further apart than this start a new recording session
NEAR_SECONDS         = 2     # a test frame within 2 s of a train frame of the same class = near-duplicate
GRADCAM_N            = 200   # test images per task used for the attention statistics
PRETRAINED           = True  # ImageNet weights (needs Internet = On)

DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP     = DEVICE.type == "cuda"
NUM_WORKERS = min(4, os.cpu_count() or 2)
torch.backends.cudnn.benchmark = True

if SMOKE_TEST:
    EPOCHS, SEEDS, SIMCLR_EPOCHS, GRADCAM_N = 1, SEEDS[:1], 1, 8

BASE_OUT     = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
OUT_DIR      = os.path.join(BASE_OUT, f"paper_fixes_{SPLIT_MODE}" + ("_smoke" if SMOKE_TEST else ""))
os.makedirs(OUT_DIR, exist_ok=True)
RUNS_FILE    = os.path.join(OUT_DIR, "runs.jsonl")
SIMCLR_PATH  = os.path.join(OUT_DIR, "simclr_backbone.pth")
FULL_WEIGHTS = os.path.join(OUT_DIR, f"full_model_seed{SEEDS[0]}.pth")

print(f"Dataset : {DATASET_ROOT}")
print(f"Output  : {OUT_DIR}")
print(f"Device  : {DEVICE} | mixed precision: {USE_AMP} | data workers: {NUM_WORKERS}")
print(f"Plan    : split={SPLIT_MODE} | seeds={SEEDS} | epochs={EPOCHS} | smoke test={SMOKE_TEST}")

In [ ]:
TS_RE = re.compile(r"(\d{8})[_\-]?(\d{6})")

def parse_ts(stem):
    """'img20231126_200643' -> POSIX seconds (None if the name has no timestamp)."""
    m = TS_RE.search(stem)
    if not m:
        return None
    try:
        return datetime.strptime(m.group(1) + m.group(2), "%Y%m%d%H%M%S").timestamp()
    except ValueError:
        return None

def canon(name, options, aliases=None):
    key = name.strip().lower()
    if aliases:
        key = aliases.get(key, key)
    for o in options:
        if o.lower() == key:
            return o
    return None

def scan(root, task):
    out, skipped = [], set()
    for cdir in sorted(os.listdir(root)):
        cpath = os.path.join(root, cdir)
        if not os.path.isdir(cpath):
            continue
        cls = canon(cdir, EMOTIONS, EMO_ALIASES) if task == "emo" else (cdir.strip() if cdir.strip().isdigit() else None)
        if cls is None:
            skipped.add(cdir); continue
        for pdir in sorted(os.listdir(cpath)):
            ppath = os.path.join(cpath, pdir)
            if not os.path.isdir(ppath):
                continue
            pal = canon(pdir, PALETTES)
            if pal is None:
                skipped.add(f"{cdir}/{pdir}"); continue
            for fn in sorted(os.listdir(ppath)):
                if fn.lower().endswith(IMG_EXTS):
                    full = os.path.join(ppath, fn)
                    stem = os.path.splitext(fn)[0]
                    out.append({"path": full, "rel": os.path.relpath(full, DATASET_ROOT), "task": task,
                                "cls": cls, "pal": pal, "stem": stem, "ts": parse_ts(stem)})
    if skipped:
        print(f"  ⚠ ignored folders in {os.path.basename(root)}: {sorted(skipped)[:12]}")
    return out

emo_samples = scan(EMOTION_ROOT, "emo")
id_samples  = scan(IDENTITY_ROOT, "id")
assert emo_samples and id_samples, "No images found — check the dataset folders."

PERSONS    = sorted({s["cls"] for s in id_samples}, key=int)
EMO_TO_IDX = {e: i for i, e in enumerate(EMOTIONS)}
ID_TO_IDX  = {p: i for i, p in enumerate(PERSONS)}
PAL_TO_IDX = {p: i for i, p in enumerate(PALETTES)}
NUM_EMO, NUM_ID, NUM_PAL = len(EMOTIONS), len(PERSONS), len(PALETTES)

SAMPLES = emo_samples + id_samples
for s in SAMPLES:
    s["emo_label"] = EMO_TO_IDX[s["cls"]] if s["task"] == "emo" else -1
    s["id_label"]  = ID_TO_IDX[s["cls"]] if s["task"] == "id" else -1
    s["pal_idx"]   = PAL_TO_IDX[s["pal"]]

counts = pd.crosstab(pd.Series(["Emotion" if s["task"] == "emo" else "Identity" for s in SAMPLES], name="task"),
                     pd.Series([s["pal"] for s in SAMPLES], name="palette"))
print(f"Emotion images : {len(emo_samples)} | classes: {EMOTIONS}")
print(f"Identity images: {len(id_samples)} | persons: {PERSONS}")
print(counts.to_string())

## 3 · Fixed split + leakage check

The split is made **once** (seed 42), saved to `split_<mode>.json`, and reused by every run and every seed — so differences between configurations are not split noise.

The leakage check looks for two things a frame-level random split can put on both sides: the **same frame rendered in another palette**, and **near-identical consecutive frames** (≤ 2 s apart). If the verdict is HIGH, test accuracy measures recognition of frames the model has effectively seen. `"frame"` removes the first kind; `"session"` removes both, but if a class was recorded in only a few sessions its split can be lopsided (check the printed split sizes).

In [ ]:
def assign_sessions(samples):
    """Group frames of the same class into recording sessions by timestamp gaps (all palettes together)."""
    by_cls = defaultdict(list)
    for i, s in enumerate(samples):
        by_cls[(s["task"], s["cls"])].append(i)
    for (task, cls), idxs in by_cls.items():
        timed = sorted((i for i in idxs if samples[i]["ts"] is not None), key=lambda i: samples[i]["ts"])
        sid, prev = 0, None
        for i in timed:
            if prev is not None and samples[i]["ts"] - prev > GAP_SECONDS:
                sid += 1
            samples[i]["session"] = f"{task}|{cls}|s{sid}"
            prev = samples[i]["ts"]
        for i in idxs:
            if samples[i]["ts"] is None:
                samples[i]["session"] = f"{task}|{cls}|f{samples[i]['stem']}"

def make_split(samples, mode, seed):
    rng, split, notes = random.Random(seed), {}, []
    by_cls = defaultdict(list)
    for i, s in enumerate(samples):
        by_cls[(s["task"], s["cls"])].append(i)

    def random_split(idxs):            # the paper's protocol: stratified by class × palette
        by_pal = defaultdict(list)
        for i in idxs:
            by_pal[samples[i]["pal"]].append(i)
        for pal in sorted(by_pal):
            b = sorted(by_pal[pal]); rng.shuffle(b); m = len(b)
            nv = max(1, int(m * VAL_R)); nt = max(1, int(m * TEST_R)) if m > 3 else 0
            for i in b[:nv]: split[i] = "val"
            for i in b[nv:nv + nt]: split[i] = "test"
            for i in b[nv + nt:]: split[i] = "train"

    for key in sorted(by_cls):
        idxs = by_cls[key]
        if mode == "random":
            random_split(idxs); continue
        field = "stem" if mode == "frame" else "session"
        groups = defaultdict(list)
        for i in idxs:
            groups[samples[i][field]].append(i)
        keys = sorted(groups); rng.shuffle(keys)
        if len(keys) < 3:
            notes.append(f"{key[0]}:{key[1]} has only {len(keys)} {field} group(s) → random split used for this class")
            random_split(idxs); continue
        n, pos, parts = len(idxs), 0, {"test": [], "val": [], "train": []}
        for part, frac in (("test", TEST_R), ("val", VAL_R)):
            got = 0
            while pos < len(keys) - 1 and (got == 0 or got < frac * n):
                parts[part].append(keys[pos]); got += len(groups[keys[pos]]); pos += 1
        parts["train"] = keys[pos:]
        for part, ks in parts.items():
            for k in ks:
                for i in groups[k]:
                    split[i] = part
    return split, notes

assign_sessions(SAMPLES)
SPLIT, SPLIT_NOTES = make_split(SAMPLES, SPLIT_MODE, SPLIT_SEED)
split_file = os.path.join(OUT_DIR, f"split_{SPLIT_MODE}.json")
rel_to_i = {s["rel"]: i for i, s in enumerate(SAMPLES)}
if os.path.exists(split_file):
    with open(split_file) as fh:
        saved = json.load(fh)
    if set(saved) == set(rel_to_i):
        SPLIT = {rel_to_i[r]: p for r, p in saved.items()}
        print(f"✓ re-using the saved split ({split_file})")
    else:
        print("⚠ the saved split does not match the current files — a new split was made")
        with open(split_file, "w") as fh:
            json.dump({SAMPLES[i]["rel"]: p for i, p in SPLIT.items()}, fh)
else:
    with open(split_file, "w") as fh:
        json.dump({SAMPLES[i]["rel"]: p for i, p in SPLIT.items()}, fh)
for n_ in SPLIT_NOTES:
    print("  note:", n_)

def idx_of(task, part):
    return [i for i in range(len(SAMPLES)) if SAMPLES[i]["task"] == task and SPLIT.get(i) == part]

EMO_TR, EMO_VA, EMO_TE = (idx_of("emo", p) for p in ("train", "val", "test"))
ID_TR,  ID_VA,  ID_TE  = (idx_of("id", p) for p in ("train", "val", "test"))
if SMOKE_TEST:
    _r = random.Random(0)
    def _sub(lst, k):
        lst = lst[:]; _r.shuffle(lst); return sorted(lst[:k])
    EMO_TR, ID_TR = _sub(EMO_TR, 96), _sub(ID_TR, 96)
    EMO_VA, ID_VA, EMO_TE, ID_TE = (_sub(l, 40) for l in (EMO_VA, ID_VA, EMO_TE, ID_TE))
TRAIN_IDX, VAL_IDX, TEST_IDX = EMO_TR + ID_TR, EMO_VA + ID_VA, EMO_TE + ID_TE

print(f"\nSplit sizes ({SPLIT_MODE}):")
print(f"  Emotion  train {len(EMO_TR):5d} | val {len(EMO_VA):4d} | test {len(EMO_TE):4d}")
print(f"  Identity train {len(ID_TR):5d} | val {len(ID_VA):4d} | test {len(ID_TE):4d}")

def leakage_report():
    rows = {}
    for task, tr, te, name in (("emo", EMO_TR, EMO_TE, "Emotion"), ("id", ID_TR, ID_TE, "Identity")):
        idx = [i for i, s in enumerate(SAMPLES) if s["task"] == task]
        pals = defaultdict(set)
        for i in idx:
            pals[(SAMPLES[i]["cls"], SAMPLES[i]["stem"])].add(SAMPLES[i]["pal"])
        shared = np.mean([len(pals[(SAMPLES[i]["cls"], SAMPLES[i]["stem"])]) > 1 for i in idx])
        ts_ok = np.mean([SAMPLES[i]["ts"] is not None for i in idx])
        sess = defaultdict(set)
        for i in idx:
            sess[SAMPLES[i]["cls"]].add(SAMPLES[i]["session"])
        n_sess = [len(v) for v in sess.values()]
        train_frames = {(SAMPLES[i]["cls"], SAMPLES[i]["stem"]) for i in tr}
        twin = np.mean([(SAMPLES[i]["cls"], SAMPLES[i]["stem"]) in train_frames for i in te]) if te else float("nan")
        train_ts = defaultdict(list)
        for i in tr:
            if SAMPLES[i]["ts"] is not None:
                train_ts[SAMPLES[i]["cls"]].append(SAMPLES[i]["ts"])
        for v in train_ts.values():
            v.sort()
        near = []
        for i in te:
            t, arr = SAMPLES[i]["ts"], train_ts.get(SAMPLES[i]["cls"], [])
            if t is None or not arr:
                continue
            j = bisect.bisect_left(arr, t)
            near.append(min(abs(arr[k] - t) for k in (j - 1, j) if 0 <= k < len(arr)) <= NEAR_SECONDS)
        rows[name] = {"images": len(idx),
                      "frames also present in another palette": f"{shared * 100:.1f}%",
                      "file names with a timestamp": f"{ts_ok * 100:.1f}%",
                      f"recording sessions per class (gap > {GAP_SECONDS}s)": f"{min(n_sess)}–{max(n_sess)}",
                      "TEST frames whose other-palette twin is in TRAIN": f"{twin * 100:.1f}%",
                      f"TEST frames with a TRAIN frame ≤{NEAR_SECONDS}s away (same class)": (f"{np.mean(near) * 100:.1f}%" if near else "n/a"),
                      "_risk": max(twin if twin == twin else 0, float(np.mean(near)) if near else 0)}
    risk = max(r.pop("_risk") for r in rows.values())
    verdict = ("HIGH" if risk >= 0.5 else "MODERATE" if risk >= 0.1 else "LOW")
    df = pd.DataFrame(rows)
    text = (f"LEAKAGE CHECK (split = {SPLIT_MODE})\n{df.to_string()}\n\nVerdict: {verdict} "
            f"(worst rate {risk * 100:.1f}%).\n")
    advice = {"random": "Compare with SPLIT_MODE = 'frame' (removes other-palette copies) or 'session' "
                        "(also removes neighbouring frames).",
              "frame": "'frame' removed the other-palette copies; neighbouring frames of the same recording remain. "
                       "Only SPLIT_MODE = 'session' removes those.",
              "session": "Some leakage remains even with whole sessions held out — check GAP_SECONDS."}[SPLIT_MODE]
    if verdict == "HIGH":
        text += ("→ Most test frames have a copy (other palette) or a near-identical neighbour in TRAIN, so test accuracy "
                 "mostly measures recognition of frames the model has effectively seen, not generalisation to new "
                 f"recordings. {advice} Otherwise, state this as a limitation in the paper.\n")
    elif verdict == "MODERATE":
        text += f"→ Some test frames have close copies in TRAIN. Report this as a limitation. {advice}\n"
    else:
        text += "→ No sign of frame-level leakage under this split.\n"
    return text, verdict

LEAK_TEXT, LEAK_VERDICT = leakage_report()
open(os.path.join(OUT_DIR, "leakage_report.txt"), "w").write(LEAK_TEXT)
print("\n" + LEAK_TEXT)

## 4 · Images, augmentation and loaders

Images are loaded **in colour**, so the false-colour palette is kept (the original notebook loaded grayscale, which removed most palette differences). Augmentation is identical to your earlier Kaggle runs.

In [ ]:
NORM = dict(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))

def train_tf():
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.Affine(translate_percent=(-0.05, 0.05), scale=(0.90, 1.10), rotate=(-10, 10),
                 border_mode=cv2.BORDER_CONSTANT, p=0.5),
        A.GaussNoise(std_range=(0.01, 0.05), p=0.35),
        A.GaussianBlur(blur_limit=(3, 5), p=0.20),
        A.RandomBrightnessContrast(brightness_limit=0.30, contrast_limit=0.30, p=0.40),
        A.CLAHE(clip_limit=2.0, tile_grid_size=(4, 4), p=0.20),
        A.CoarseDropout(num_holes_range=(1, 3), hole_height_range=(0.05, 0.12),
                        hole_width_range=(0.05, 0.12), fill=0, p=0.20),
        A.Resize(IMG_SIZE, IMG_SIZE), A.Normalize(**NORM), ToTensorV2()])

def val_tf():
    return A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.Normalize(**NORM), ToTensorV2()])

def simclr_tf():
    return A.Compose([
        A.RandomResizedCrop(size=(IMG_SIZE, IMG_SIZE), scale=(0.6, 1.0)),
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.4, contrast_limit=0.4, p=0.5),
        A.GaussNoise(std_range=(0.01, 0.06), p=0.4),
        A.GaussianBlur(blur_limit=(3, 7), p=0.3),
        A.CLAHE(clip_limit=3.0, p=0.3),
        A.Normalize(**NORM), ToTensorV2()])

def load_rgb(path):
    img = cv2.imread(path, cv2.IMREAD_COLOR)       # keeps the false-colour palette
    if img is None:
        img = np.zeros((IMG_SIZE, IMG_SIZE, 3), np.uint8)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

class ThermalDS(Dataset):
    def __init__(self, indices, tf):
        self.indices, self.tf = list(indices), tf
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, k):
        i = self.indices[k]; s = SAMPLES[i]
        return {"image": self.tf(image=load_rgb(s["path"]))["image"],
                "emo": s["emo_label"], "id": s["id_label"], "pal": s["pal_idx"], "i": i}

class BalancedBatchSampler(Sampler):
    """Every batch = half emotion images + half identity images (global sample indices)."""
    def __init__(self, emo_idx, id_idx, batch_size, seed):
        self.e, self.d, self.half = list(emo_idx), list(id_idx), batch_size // 2
        self.rng = random.Random(seed)
        self.n_batches = max(1, max(len(self.e), len(self.d)) // self.half)
    def __iter__(self):
        e, d = self.e[:], self.d[:]
        self.rng.shuffle(e); self.rng.shuffle(d)
        for b in range(self.n_batches):
            s = b * self.half
            yield [e[(s + k) % len(e)] for k in range(self.half)] + [d[(s + k) % len(d)] for k in range(self.half)]
    def __len__(self):
        return self.n_batches

def make_train_loader(tf, seed):
    return DataLoader(ThermalDS(range(len(SAMPLES)), tf),
                      batch_sampler=BalancedBatchSampler(EMO_TR, ID_TR, BATCH_SIZE, seed),
                      num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda",
                      persistent_workers=NUM_WORKERS > 0)

def make_eval_loader(indices, persistent=False):
    return DataLoader(ThermalDS(indices, val_tf()), batch_size=64, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda",
                      persistent_workers=persistent and NUM_WORKERS > 0)

_cnt = np.bincount([SAMPLES[i]["emo_label"] for i in EMO_TR], minlength=NUM_EMO).astype(float)
_w = 1.0 / np.maximum(_cnt, 1)
EMO_CLASS_W = torch.tensor(_w / _w.sum() * NUM_EMO, dtype=torch.float32, device=DEVICE)
print("✓ loaders ready | focal-loss class weights:", np.round(EMO_CLASS_W.cpu().numpy(), 3))

## 5 · Model: EfficientNet-B0 multi-task network with GRL (N1), PaIN (N8) and margin classifiers

In [ ]:
class GradReverseFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.clone()
    @staticmethod
    def backward(ctx, g):
        return -ctx.alpha * g, None

class GradReverse(nn.Module):                      # N1
    def __init__(self):
        super().__init__(); self.alpha = 0.0
    def forward(self, x):
        return GradReverseFn.apply(x, self.alpha)

class PaIN(nn.Module):                             # N8: palette-specific running statistics
    def __init__(self, C, eps=1e-5, momentum=0.1):
        super().__init__()
        self.eps, self.mom = eps, momentum
        self.gamma, self.beta = nn.Parameter(torch.ones(C)), nn.Parameter(torch.zeros(C))
        self.register_buffer("run_mean", torch.zeros(NUM_PAL, C))
        self.register_buffer("run_var", torch.ones(NUM_PAL, C))
    def forward(self, x, pal_idx):
        if pal_idx is None:
            return x
        if self.training:
            with torch.no_grad():
                for p in pal_idx.unique():
                    m = pal_idx == p
                    if m.sum() < 2:
                        continue
                    xp = x[m].float()
                    self.run_mean[p] = (1 - self.mom) * self.run_mean[p] + self.mom * xp.mean(0)
                    self.run_var[p] = (1 - self.mom) * self.run_var[p] + self.mom * xp.var(0, unbiased=False)
        return self.gamma * (x - self.run_mean[pal_idx]) / (self.run_var[pal_idx] + self.eps).sqrt() + self.beta

class MarginClassifier(nn.Module):
    """Cosine classifier.  forward() -> s·cos (used for PREDICTION);  margin_logits() -> ArcFace/CosFace
    logits (used for TRAINING).  Both use the same weight matrix — this is the fix for the 2.43% ArcFace bug."""
    def __init__(self, dim, n_cls, kind, s, m):
        super().__init__()
        self.kind, self.s, self.m = kind, s, m
        self.W = nn.Parameter(torch.empty(n_cls, dim)); nn.init.xavier_uniform_(self.W)
        self.cos_m, self.sin_m = math.cos(m), math.sin(m)
        self.th, self.mm = math.cos(math.pi - m), math.sin(math.pi - m) * m
    def cosine(self, feat):
        with torch.autocast(device_type=feat.device.type, enabled=False):
            return F.linear(F.normalize(feat.float(), dim=1), F.normalize(self.W.float(), dim=1)).clamp(-1 + 1e-7, 1 - 1e-7)
    def forward(self, feat):
        return self.s * self.cosine(feat)
    def margin_logits(self, feat, labels):
        cos = self.cosine(feat)
        if self.kind == "arcface":
            sin = torch.sqrt((1.0 - cos * cos).clamp_min(1e-7))
            phi = torch.where(cos > self.th, cos * self.cos_m - sin * self.sin_m, cos - self.mm)
        else:                                      # cosface
            phi = cos - self.m
        onehot = F.one_hot(labels, cos.size(1)).to(cos.dtype)
        return self.s * (onehot * phi + (1 - onehot) * cos)

class ThermalMTL(nn.Module):
    FEAT, PROJ = 1280, 256
    def __init__(self, use_pain=False, id_loss="ce"):
        super().__init__()
        base = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT if PRETRAINED else None)
        self.backbone, self.pool = base.features, nn.AdaptiveAvgPool2d(1)
        self.use_pain, self.pain = use_pain, PaIN(self.FEAT)
        self.emo_proj = nn.Sequential(nn.Linear(self.FEAT, self.PROJ), nn.BatchNorm1d(self.PROJ))
        self.id_proj  = nn.Sequential(nn.Linear(self.FEAT, self.PROJ), nn.BatchNorm1d(self.PROJ))
        self.emo_head = nn.Sequential(nn.ReLU(), nn.Dropout(0.3), nn.Linear(self.PROJ, NUM_EMO))
        if id_loss in ("arcface", "cosface"):
            s, m = (ARC_S, ARC_M) if id_loss == "arcface" else (COS_S, COS_M)
            self.id_head = MarginClassifier(self.PROJ, NUM_ID, id_loss, s, m)
        else:
            self.id_head = nn.Sequential(nn.ReLU(), nn.Dropout(0.3), nn.Linear(self.PROJ, NUM_ID))
        self.grl = GradReverse()
        self.palette_head = nn.Sequential(nn.Linear(self.FEAT, 128), nn.ReLU(), nn.Dropout(0.3),
                                          nn.Linear(128, NUM_PAL))
    def encode(self, x, pal_idx):
        f = self.pool(self.backbone(x)).flatten(1)
        fn = self.pain(f, pal_idx) if self.use_pain else f
        return f, self.emo_proj(fn), self.id_proj(fn)
    def forward(self, x, pal_idx=None):
        f, ef, idf = self.encode(x, pal_idx)
        return self.emo_head(ef), self.id_head(idf), self.palette_head(self.grl(f)), ef, idf

print("✓ model defined")

## 6 · Losses: focal (A1a), orthogonality (N2, bug fixed), CLUB, SupCon

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None, smooth=0.1):
        super().__init__(); self.gamma, self.alpha, self.smooth = gamma, alpha, smooth
    def forward(self, logits, y):
        logits = logits.float(); C = logits.size(1)
        with torch.no_grad():
            t = torch.full_like(logits, self.smooth / (C - 1)); t.scatter_(1, y[:, None], 1 - self.smooth)
        logp = F.log_softmax(logits, 1)
        loss = (1 - (logp.exp() * t).sum(1)) ** self.gamma * -(t * logp).sum(1)
        if self.alpha is not None:
            loss = self.alpha[y] * loss
        return loss.mean()

def ortho_loss(ef, idf):
    """Paper Eq. (3): mean squared cosine between each image's emotion and identity projections.
    Applied to EVERY image — the original code only applied it to images with both labels (there are none)."""
    return F.cosine_similarity(ef.float(), idf.float(), dim=1, eps=1e-8).pow(2).mean()

class CLUB(nn.Module):
    """Contrastive log-ratio upper bound of MI(ef; idf) (Cheng et al., ICML 2020)."""
    def __init__(self, x_dim=256, y_dim=256, hidden=256):
        super().__init__()
        self.mu = nn.Sequential(nn.Linear(x_dim, hidden), nn.ReLU(), nn.Linear(hidden, y_dim))
        self.logvar = nn.Sequential(nn.Linear(x_dim, hidden), nn.ReLU(), nn.Linear(hidden, y_dim), nn.Tanh())
    def mi_est(self, x, y):
        mu, logvar = self.mu(x), self.logvar(x)
        pos = -0.5 * (((mu - y) ** 2) / logvar.exp()).sum(1)
        neg = -0.5 * (((mu - y[torch.randperm(y.size(0), device=y.device)]) ** 2) / logvar.exp()).sum(1)
        return (pos - neg).mean()
    def learning_loss(self, x, y):
        mu, logvar = self.mu(x), self.logvar(x)
        return 0.5 * (((mu - y) ** 2) / logvar.exp() + logvar).sum(1).mean()

def supcon_loss(z, y, T=SUPCON_T):
    """Supervised contrastive loss (Khosla et al., NeurIPS 2020) on L2-normalised embeddings."""
    n = z.size(0)
    if n < 2:
        return z.sum() * 0
    not_self = ~torch.eye(n, dtype=torch.bool, device=z.device)
    pos = (y[:, None] == y[None, :]) & not_self
    sim = z @ z.T / T
    sim = sim - sim.max(1, keepdim=True).values.detach()
    log_prob = sim - torch.log((sim.exp() * not_self).sum(1, keepdim=True) + 1e-12)
    npos = pos.sum(1)
    valid = npos > 0
    if not valid.any():
        return z.sum() * 0
    return -((log_prob * pos).sum(1)[valid] / npos[valid]).mean()

print("✓ losses defined")

## 7 · Training and evaluation

Every configuration: AdamW (lr 3e-4, wd 1e-4), cosine schedule, `EPOCHS` epochs, mixed precision on GPU, best epoch chosen on the **validation** split (mean of emotion and identity accuracy), then evaluated once on the **test** split.

In [ ]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def amp_ctx():
    return torch.autocast(device_type="cuda", dtype=torch.float16) if USE_AMP else nullcontext()

def make_scaler():
    try:
        return torch.amp.GradScaler("cuda", enabled=USE_AMP)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=USE_AMP)

@torch.no_grad()
def infer_loader(model, loader, feats=False):
    model.eval(); out = defaultdict(list)
    for b in loader:
        x, p = b["image"].to(DEVICE, non_blocking=True), b["pal"].to(DEVICE)
        with amp_ctx():
            f, ef, idf = model.encode(x, p)
            eo, io = model.emo_head(ef), model.id_head(idf)
        out["emo_pred"].append(eo.argmax(1).cpu()); out["id_pred"].append(io.argmax(1).cpu())
        for k in ("emo", "id", "pal", "i"):
            out[k].append(b[k])
        if feats:
            out["f"].append(f.float().cpu()); out["ef"].append(ef.float().cpu()); out["idf"].append(idf.float().cpu())
    return {k: torch.cat(v).numpy() for k, v in out.items()}

def infer(model, indices, feats=False):
    return infer_loader(model, make_eval_loader(indices), feats)

def cls_metrics(y, yhat):
    if len(y) == 0:
        return {"acc": float("nan"), "prec": float("nan"), "rec": float("nan"), "f1": float("nan"), "n": 0}
    p, r, f1, _ = precision_recall_fscore_support(y, yhat, labels=np.unique(y), average="macro", zero_division=0)
    return {"acc": float(accuracy_score(y, yhat)), "prec": float(p), "rec": float(r), "f1": float(f1), "n": int(len(y))}

def evaluate(res):
    em_, id_ = res["emo"] >= 0, res["id"] >= 0
    out = {"emo": cls_metrics(res["emo"][em_], res["emo_pred"][em_]),
           "id": cls_metrics(res["id"][id_], res["id_pred"][id_]), "per_palette": {}}
    for pi, pal in enumerate(PALETTES):
        pe, pd_ = em_ & (res["pal"] == pi), id_ & (res["pal"] == pi)
        out["per_palette"][pal] = {"emo": cls_metrics(res["emo"][pe], res["emo_pred"][pe]),
                                   "id": cls_metrics(res["id"][pd_], res["id_pred"][pd_])}
    return out

def probe_acc(Xtr, ytr, Xte, yte):
    if len(np.unique(ytr)) < 2 or len(yte) == 0:
        return float("nan")
    sc = StandardScaler().fit(Xtr)
    clf = LogisticRegression(max_iter=300).fit(sc.transform(Xtr), ytr)
    return float((clf.predict(sc.transform(Xte)) == yte).mean())

def run_probes(model, te):
    """Linear probes: how much palette / cross-task information the features still carry (lower = more invariant)."""
    tr = infer(model, TRAIN_IDX, feats=True)
    out = {"cos2": float((F.cosine_similarity(torch.from_numpy(te["ef"]), torch.from_numpy(te["idf"]), dim=1) ** 2).mean())}
    out["pal_from_f"]   = probe_acc(tr["f"], tr["pal"], te["f"], te["pal"])
    out["pal_from_emo"] = probe_acc(tr["ef"], tr["pal"], te["ef"], te["pal"])
    out["pal_from_id"]  = probe_acc(tr["idf"], tr["pal"], te["idf"], te["pal"])
    a, b = tr["id"] >= 0, te["id"] >= 0
    out["id_from_emo"]  = probe_acc(tr["ef"][a], tr["id"][a], te["ef"][b], te["id"][b])
    a, b = tr["emo"] >= 0, te["emo"] >= 0
    out["emo_from_id"]  = probe_acc(tr["idf"][a], tr["emo"][a], te["idf"][b], te["emo"][b])
    return out

def train_run(cfg, seed):
    set_seed(seed)
    model = ThermalMTL(use_pain=cfg["pain"], id_loss=cfg["id_loss"]).to(DEVICE)
    if cfg["simclr"]:
        model.backbone.load_state_dict(torch.load(SIMCLR_PATH, map_location=DEVICE))
    train_loader = make_train_loader(train_tf() if cfg["aug"] else val_tf(), seed)
    val_loader = make_eval_loader(VAL_IDX, persistent=True)
    ce = nn.CrossEntropyLoss(label_smoothing=0.1)
    focal = FocalLoss(alpha=EMO_CLASS_W) if cfg["focal"] else None
    club = CLUB(ThermalMTL.PROJ, ThermalMTL.PROJ).to(DEVICE) if cfg["club"] else None
    club_opt = torch.optim.Adam(club.parameters(), lr=1e-3) if club is not None else None
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sch = CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-6)
    scaler = make_scaler()
    best, best_state, hist, t_start = -1.0, None, [], time.time()
    for ep in range(1, EPOCHS + 1):
        model.grl.alpha = (2 / (1 + math.exp(-10 * ep / EPOCHS)) - 1) * GRL_ALPHA_MAX if cfg["grl"] else 0.0
        model.train(); t0 = time.time()
        tot = n = ec = en = ic = icn = 0; mi_sum = 0.0
        for b in train_loader:
            x = b["image"].to(DEVICE, non_blocking=True)
            e, d, p = b["emo"].to(DEVICE), b["id"].to(DEVICE), b["pal"].to(DEVICE)
            with amp_ctx():
                eo, io, po, ef, idf = model(x, p)
            if club is not None:                       # (a) fit the MI estimator on detached features
                club_opt.zero_grad(set_to_none=True)
                club.learning_loss(ef.detach().float(), idf.detach().float()).backward()
                club_opt.step()
            em, im = e >= 0, d >= 0
            losses = []
            if em.any():
                losses.append(W_EMO * (focal(eo[em], e[em]) if focal is not None else ce(eo[em].float(), e[em])))
            if im.any():
                if cfg["id_loss"] in ("arcface", "cosface"):
                    losses.append(W_ID * F.cross_entropy(model.id_head.margin_logits(idf[im], d[im]), d[im]))
                else:
                    losses.append(W_ID * ce(io[im].float(), d[im]))
                    if cfg["id_loss"] == "supcon":
                        losses.append(W_SUPCON * supcon_loss(F.normalize(idf[im].float(), dim=1), d[im]))
            if cfg["grl"]:
                losses.append(W_PAL * ce(po.float(), p))
            if cfg["ortho"]:
                losses.append(W_ORT * ortho_loss(ef, idf))
            if club is not None:                       # (b) minimise the MI estimate through the network
                mi = club.mi_est(ef.float(), idf.float()); losses.append(W_CLUB * mi); mi_sum += mi.item()
            loss = sum(losses)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tot += loss.item(); n += 1
            with torch.no_grad():
                if em.any():
                    ec += (eo[em].argmax(1) == e[em]).sum().item(); en += int(em.sum())
                if im.any():
                    ic += (io[im].argmax(1) == d[im]).sum().item(); icn += int(im.sum())
        sch.step()
        v = evaluate(infer_loader(model, val_loader))
        hist.append({"ep": ep, "loss": tot / max(n, 1), "tr_emo": ec / max(en, 1), "tr_id": ic / max(icn, 1),
                     "val_emo": v["emo"]["acc"], "val_id": v["id"]["acc"],
                     "mi": (mi_sum / max(n, 1)) if club is not None else None, "sec": time.time() - t0})
        score = float(np.nan_to_num(0.5 * v["emo"]["acc"] + 0.5 * v["id"]["acc"], nan=0.0))
        if best_state is None or score > best:
            best, best_state = score, {k: t.detach().cpu().clone() for k, t in model.state_dict().items()}
        if ep == 1 or ep % 5 == 0 or ep == EPOCHS:
            h = hist[-1]
            print(f"    ep {ep:02d}/{EPOCHS} | loss {h['loss']:.3f} | val emo {h['val_emo']:.3f} id {h['val_id']:.3f}"
                  + (f" | MI {h['mi']:+.3f}" if club is not None else "") + f" | {h['sec']:.0f}s")
    model.load_state_dict(best_state)
    del train_loader, val_loader
    return model, hist, (time.time() - t_start) / 60

print("✓ training / evaluation functions defined")

## 8 · SimCLR pre-training (N7) — uses TRAIN images only

In [ ]:
class SimCLRNet(nn.Module):
    def __init__(self):
        super().__init__()
        base = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT if PRETRAINED else None)
        self.backbone, self.pool = base.features, nn.AdaptiveAvgPool2d(1)
        self.proj = nn.Sequential(nn.Linear(1280, 512), nn.BatchNorm1d(512), nn.ReLU(inplace=True), nn.Linear(512, 128))
    def forward(self, x):
        return F.normalize(self.proj(self.pool(self.backbone(x)).flatten(1)).float(), dim=1)

class TwoViewDS(Dataset):
    def __init__(self, indices):
        self.indices, self.tf = list(indices), simclr_tf()
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, k):
        img = load_rgb(SAMPLES[self.indices[k]]["path"])
        return self.tf(image=img)["image"], self.tf(image=img)["image"]

def nt_xent(z1, z2, tau=SIMCLR_TAU):
    z, B = torch.cat([z1, z2]), z1.size(0)
    sim = z @ z.T / tau - torch.eye(2 * B, device=z.device) * 1e9
    target = torch.cat([torch.arange(B, 2 * B), torch.arange(B)]).to(z.device)
    return F.cross_entropy(sim, target)

def pretrain_simclr():
    set_seed(SEEDS[0])
    net = SimCLRNet().to(DEVICE)
    bs = max(2, min(SIMCLR_BATCH, len(TRAIN_IDX)))
    loader = DataLoader(TwoViewDS(TRAIN_IDX), batch_size=bs, shuffle=True, drop_last=True,
                        num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda", persistent_workers=NUM_WORKERS > 0)
    opt = torch.optim.AdamW(net.parameters(), lr=3e-4, weight_decay=1e-4)
    sch = CosineAnnealingLR(opt, T_max=SIMCLR_EPOCHS, eta_min=1e-6)
    scaler = make_scaler()
    for ep in range(1, SIMCLR_EPOCHS + 1):
        net.train(); tot = n = 0
        for v1, v2 in loader:
            v1, v2 = v1.to(DEVICE, non_blocking=True), v2.to(DEVICE, non_blocking=True)
            with amp_ctx():
                z1, z2 = net(v1), net(v2)
            loss = nt_xent(z1.float(), z2.float())
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tot += loss.item(); n += 1
        sch.step()
        if ep == 1 or ep % 5 == 0 or ep == SIMCLR_EPOCHS:
            print(f"    SimCLR ep {ep:02d}/{SIMCLR_EPOCHS} | NT-Xent {tot / max(n, 1):.4f}")
    torch.save(net.backbone.state_dict(), SIMCLR_PATH)
    del loader, net
    print(f"  ✓ SimCLR backbone saved → {SIMCLR_PATH}")

print("✓ SimCLR defined")

## 9 · Run all experiments (resumable)

Runs are ordered seed by seed, so after the first ~2 h you already have one complete set. Each finished run is appended to `runs.jsonl`; re-running this cell skips finished runs.

| Configuration | Aug (N4) | GRL (N1) | PaIN (N8) | Ortho (N2) | Focal (A1a) | Extra |
|---|---|---|---|---|---|---|
| Baseline (CE only) | | | | | | |
| + X only | one of the five | | | | | |
| + GRL + PaIN | | ✓ | ✓ | | | |
| + GRL + PaIN + Ortho | | ✓ | ✓ | ✓ | | |
| Full Model | ✓ | ✓ | ✓ | ✓ | ✓ | |
| Full Model + SimCLR / + CLUB | ✓ | ✓ | ✓ | ✓ | ✓ | SimCLR init / CLUB |
| Full Model, ID loss = ArcFace / CosFace / CE+SupCon | ✓ | ✓ | ✓ | ✓ | ✓ | identity loss swapped |

In [ ]:
BASE_CFG = dict(aug=False, grl=False, pain=False, ortho=False, focal=False, club=False, simclr=False, id_loss="ce")
FULL = dict(aug=True, grl=True, pain=True, ortho=True, focal=True)
def cfg(**kw):
    c = dict(BASE_CFG); c.update(kw); return c

EXPERIMENTS = [
    ("Baseline (CE only)",           cfg(),                                "ablation"),
    ("+ Thermal Aug (N4) only",      cfg(aug=True),                        "ablation"),
    ("+ GRL (N1) only",              cfg(grl=True),                        "ablation"),
    ("+ PaIN (N8) only",             cfg(pain=True),                       "ablation"),
    ("+ Orthogonality (N2) only",    cfg(ortho=True),                      "ablation"),
    ("+ Focal Loss (A1a) only",      cfg(focal=True),                      "ablation"),
    ("+ GRL + PaIN",                 cfg(grl=True, pain=True),             "ablation"),
    ("+ GRL + PaIN + Ortho",         cfg(grl=True, pain=True, ortho=True), "ablation"),
    ("Full Model",                   cfg(**FULL),                          "ablation"),
]
if RUN_SIMCLR:
    EXPERIMENTS.append(("Full Model + SimCLR init (N7)", cfg(**FULL, simclr=True), "extension"))
if RUN_CLUB:
    EXPERIMENTS.append(("Full Model + CLUB", cfg(**FULL, club=True), "extension"))
if RUN_LOSSES:
    EXPERIMENTS += [("Full Model, ID loss = ArcFace",     cfg(**FULL, id_loss="arcface"), "loss"),
                    ("Full Model, ID loss = CosFace",     cfg(**FULL, id_loss="cosface"), "loss"),
                    ("Full Model, ID loss = CE + SupCon", cfg(**FULL, id_loss="supcon"),  "loss")]

def load_runs():
    if not os.path.exists(RUNS_FILE):
        return []
    return [json.loads(l) for l in open(RUNS_FILE) if l.strip()]

done = {(r["config"], r["seed"]) for r in load_runs() if r.get("epochs") == EPOCHS and r.get("split") == SPLIT_MODE}
todo = [(seed, name, c, cat) for seed in SEEDS for name, c, cat in EXPERIMENTS if (name, seed) not in done]
print(f"{len(EXPERIMENTS) * len(SEEDS)} runs planned | {len(EXPERIMENTS) * len(SEEDS) - len(todo)} already done | {len(todo)} to run")

for k, (seed, name, c, cat) in enumerate(todo, 1):
    print(f"\n[{k}/{len(todo)}] {name} | seed {seed}")
    try:
        if c["simclr"] and not os.path.exists(SIMCLR_PATH):
            print("  SimCLR pre-training on TRAIN images only ...")
            pretrain_simclr()
        model, hist, minutes = train_run(c, seed)
        want_probes = RUN_PROBES and cat != "loss"
        te = infer(model, TEST_IDX, feats=want_probes)
        rec = {"config": name, "category": cat, "seed": seed, "epochs": EPOCHS, "split": SPLIT_MODE,
               "cfg": c, "train_min": round(minutes, 2), **evaluate(te), "hist": hist}
        if want_probes:
            rec["probes"] = run_probes(model, te)
        with open(RUNS_FILE, "a") as fh:
            fh.write(json.dumps(rec) + "\n")
        if name == "Full Model" and seed == SEEDS[0]:
            torch.save(model.state_dict(), FULL_WEIGHTS)
        print(f"  TEST emotion {rec['emo']['acc'] * 100:.2f}% (F1 {rec['emo']['f1'] * 100:.2f}) | "
              f"identity {rec['id']['acc'] * 100:.2f}% (F1 {rec['id']['f1'] * 100:.2f}) | {minutes:.1f} min")
        if k == 1 and len(todo) > 1:
            print(f"  ⏱ rough estimate for the remaining {len(todo) - 1} runs: ~{minutes * (len(todo) - 1) / 60:.1f} h")
        del model
    except Exception:
        print(f"  ✗ {name} (seed {seed}) failed — continuing with the next run:")
        traceback.print_exc()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

print("\n✓ experiment loop finished")

## 10 · Results tables (mean ± std over seeds) → CSV, LaTeX and Markdown

Δ columns are **paired** differences (same split, same seed), which removes most split noise. With 3 seeds, treat any difference smaller than its ± as "no measurable effect".

In [ ]:
runs = [r for r in load_runs() if r.get("epochs") == EPOCHS and r.get("split") == SPLIT_MODE]
by = defaultdict(dict)
for r in runs:
    by[r["config"]][r["seed"]] = r
ORDER_ABL  = [n for n, c, cat in EXPERIMENTS if cat in ("ablation", "extension") and n in by]
N_SEEDS = len({r["seed"] for r in runs})
SEED_TXT = "single seed" if N_SEEDS == 1 else f"mean $\\pm$ std over {N_SEEDS} seeds"
EP_TXT = f"{EPOCHS} epoch" + ("s" if EPOCHS != 1 else "")
ORDER_LOSS = [n for n in ["Full Model"] + [n for n, c, cat in EXPERIMENTS if cat == "loss"] if n in by]
BASELINE = "Baseline (CE only)"

def g(path):
    def get(r):
        x = r
        for k in path.split("."):
            x = x.get(k) if isinstance(x, dict) else None
        return x
    return get

def clean(v, scale):
    return np.array([x for x in v if x is not None and x == x], dtype=float) * scale

def vals(name, getter):
    return [getter(r) for _, r in sorted(by.get(name, {}).items())]

def fmt(v, scale=100, nd=2):
    v = clean(v, scale)
    if len(v) == 0: return "---"
    if len(v) == 1: return f"{v[0]:.{nd}f}"
    return f"{v.mean():.{nd}f} ± {v.std(ddof=1):.{nd}f}"

def dlt(name, ref, getter, scale=100, nd=2):
    a, b = by.get(name, {}), by.get(ref, {})
    if name == ref:
        return "---"
    d = clean([getter(a[s]) - getter(b[s]) for s in sorted(set(a) & set(b))
               if getter(a[s]) is not None and getter(b[s]) is not None], scale)
    if len(d) == 0: return "---"
    if len(d) == 1: return f"{d[0]:+.{nd}f}"
    return f"{d.mean():+.{nd}f} ± {d.std(ddof=1):.{nd}f}"

def mean_of(name, getter, scale=100):
    v = clean(vals(name, getter), scale)
    return float(v.mean()) if len(v) else float("nan")

TEX_HEAD = {"Emo Acc": r"Emo Acc (\%)", "Emo F1": r"Emo F1 (\%)", "ID Acc": r"ID Acc (\%)", "ID F1": r"ID F1 (\%)",
            "ID Prec": r"ID Prec (\%)", "ID Rec": r"ID Rec (\%)",
            "ΔEmo vs baseline": r"$\Delta$Emo", "ΔID Acc vs CE": r"$\Delta$ID Acc",
            "cos² (ef, idf) ×100": r"cos$^2$($f_e$,$f_i$)$\times$100",
            "Palette ← f": r"Pal.\ from $f$", "Palette ← ef": r"Pal.\ from $f_e$", "Palette ← idf": r"Pal.\ from $f_i$",
            "ID ← ef": r"ID from $f_e$", "Emotion ← idf": r"Emo.\ from $f_i$",
            "Emotion head": r"Emotion head (\%)", "Identity head": r"Identity head (\%)"}

def tex_cell(x):
    return (str(x).replace("±", r"$\pm$").replace("%", r"\%").replace("_", r"\_")
            .replace("λ", r"$\lambda$").replace("→", r"$\rightarrow$"))

def write_tex(df, caption, label, fname, wide=False):
    env = "table*" if wide else "table"
    cols = list(df.columns)
    lines = [rf"\begin{{{env}}}[!t]", r"\centering", rf"\caption{{{caption}}}", rf"\label{{{label}}}",
             r"\setlength{\tabcolsep}{3pt}", r"\resizebox{" + (r"\textwidth" if wide else r"\columnwidth") + "}{!}{%",
             r"\begin{tabular}{l" + "r" * (len(cols) - 1) + "}", r"\toprule",
             " & ".join(r"\textbf{" + TEX_HEAD.get(c, tex_cell(c)) + "}" for c in cols) + r" \\", r"\midrule"]
    lines += [" & ".join(tex_cell(x) for x in row) + r" \\" for row in df.itertuples(index=False)]
    lines += [r"\bottomrule", r"\end{tabular}}", rf"\end{{{env}}}"]
    tex = "\n".join(lines) + "\n"
    open(os.path.join(OUT_DIR, fname), "w").write(tex)
    return tex

def md_table(df):
    cols = [str(c) for c in df.columns]
    rows = ["| " + " | ".join(cols) + " |", "|" + "|".join(["---"] * len(cols)) + "|"]
    rows += ["| " + " | ".join(str(x) for x in row) + " |" for row in df.itertuples(index=False)]
    return "\n".join(rows)

SUMMARY = [f"# Paper-fix results\n\nSplit: **{SPLIT_MODE}** · epochs: **{EPOCHS}** · seeds found: "
           f"**{sorted({r['seed'] for r in runs})}** · generated {datetime.now():%Y-%m-%d %H:%M}\n",
           "## Leakage check\n```\n" + LEAK_TEXT + "```\n"]

# ── Table A · ablation (+ extensions) ───────────────────────────────────────
abl = pd.DataFrame([{"Configuration": n, "Seeds": len(by[n]),
                     "Emo Acc": fmt(vals(n, g("emo.acc"))), "ΔEmo vs baseline": dlt(n, BASELINE, g("emo.acc")),
                     "Emo F1": fmt(vals(n, g("emo.f1"))), "ID Acc": fmt(vals(n, g("id.acc"))),
                     "ID F1": fmt(vals(n, g("id.f1")))} for n in ORDER_ABL])
if len(abl):
    abl.to_csv(os.path.join(OUT_DIR, "table_ablation.csv"), index=False)
    write_tex(abl, f"Ablation on EfficientNet-B0 (one fixed split, {EP_TXT}, {SEED_TXT}). "
                   f"$\\Delta$Emo is the paired difference to the baseline.",
              "tab:ablation_fixed", "table_ablation.tex", wide=True)
    print("TABLE A — ablation\n" + abl.to_string(index=False))
    SUMMARY += ["## A · Ablation (+ SimCLR / CLUB)\n", md_table(abl), ""]

# ── Table B · mechanism probes ──────────────────────────────────────────────
mech_rows = [n for n in ORDER_ABL if any("probes" in r for r in by[n].values())]
mech = pd.DataFrame([{"Configuration": n,
                      "cos² (ef, idf) ×100": fmt(vals(n, g("probes.cos2")), 100, 2),
                      "Palette ← f": fmt(vals(n, g("probes.pal_from_f")), 100, 1),
                      "Palette ← ef": fmt(vals(n, g("probes.pal_from_emo")), 100, 1),
                      "Palette ← idf": fmt(vals(n, g("probes.pal_from_id")), 100, 1),
                      "ID ← ef": fmt(vals(n, g("probes.id_from_emo")), 100, 1),
                      "Emotion ← idf": fmt(vals(n, g("probes.emo_from_id")), 100, 1)} for n in mech_rows])
if len(mech):
    mech.to_csv(os.path.join(OUT_DIR, "table_mechanisms.csv"), index=False)
    write_tex(mech, "Linear-probe test accuracy (\\%) on frozen features: lower means less palette / cross-task "
                    "information (chance: palette 20, identity 10, emotion 20). cos$^2$ is the mean squared cosine "
                    "between each image's emotion and identity projections.", "tab:mechanisms", "table_mechanisms.tex", wide=True)
    print("\nTABLE B — mechanism probes (accuracy %, lower = more invariant; chance: palette 20, ID 10, emotion 20)\n"
          + mech.to_string(index=False))
    SUMMARY += ["## B · Mechanism probes (lower = more invariant)\n", md_table(mech), ""]

# ── Table C · identity-loss comparison ──────────────────────────────────────
LOSS_LABEL = {"Full Model": "CE (Full Model)",
              "Full Model, ID loss = ArcFace": f"ArcFace (s={ARC_S:g}, m={ARC_M:g})",
              "Full Model, ID loss = CosFace": f"CosFace (s={COS_S:g}, m={COS_M:g})",
              "Full Model, ID loss = CE + SupCon": f"CE + SupCon (λ={W_SUPCON:g}, T={SUPCON_T:g})"}
loss_df = pd.DataFrame([{"Identity loss": LOSS_LABEL.get(n, n), "Seeds": len(by[n]),
                         "Emo Acc": fmt(vals(n, g("emo.acc"))), "ID Acc": fmt(vals(n, g("id.acc"))),
                         "ID Prec": fmt(vals(n, g("id.prec"))), "ID Rec": fmt(vals(n, g("id.rec"))),
                         "ID F1": fmt(vals(n, g("id.f1"))), "ΔID Acc vs CE": dlt(n, "Full Model", g("id.acc"))}
                        for n in ORDER_LOSS])
if len(loss_df) > 1:
    loss_df.to_csv(os.path.join(OUT_DIR, "table_losses.csv"), index=False)
    write_tex(loss_df, "Identity-loss comparison on the Full Model (same split and seeds). ArcFace and CosFace are "
                       "evaluated with the cosine classifier they are trained with.", "tab:id_losses", "table_losses.tex")
    print("\nTABLE C — identity losses\n" + loss_df.to_string(index=False))
    SUMMARY += ["## C · Identity-loss comparison (correct evaluation)\n", md_table(loss_df), ""]

# ── Table D · cross-palette accuracy on the TEST split (Full Model) ──────────
if "Full Model" in by:
    fm = by["Full Model"]; s0 = sorted(fm)[0]
    rows = []
    for pal in PALETTES:
        rows.append({"Palette": pal, "N emo": fm[s0]["per_palette"][pal]["emo"]["n"],
                     "Emo Acc": fmt(vals("Full Model", g(f"per_palette.{pal}.emo.acc"))),
                     "Emo F1": fmt(vals("Full Model", g(f"per_palette.{pal}.emo.f1"))),
                     "N ID": fm[s0]["per_palette"][pal]["id"]["n"],
                     "ID Acc": fmt(vals("Full Model", g(f"per_palette.{pal}.id.acc"))),
                     "ID F1": fmt(vals("Full Model", g(f"per_palette.{pal}.id.f1")))})
    per_seed_mean = lambda key: [np.nanmean([r["per_palette"][p][key]["acc"] for p in PALETTES]) for r in fm.values()]
    rows.append({"Palette": "Mean", "N emo": sum(r["N emo"] for r in rows), "Emo Acc": fmt(per_seed_mean("emo")),
                 "Emo F1": fmt([np.nanmean([r["per_palette"][p]["emo"]["f1"] for p in PALETTES]) for r in fm.values()]),
                 "N ID": sum(r["N ID"] for r in rows), "ID Acc": fmt(per_seed_mean("id")),
                 "ID F1": fmt([np.nanmean([r["per_palette"][p]["id"]["f1"] for p in PALETTES]) for r in fm.values()])})
    pal_df = pd.DataFrame(rows)
    pal_df.to_csv(os.path.join(OUT_DIR, "table_cross_palette.csv"), index=False)
    write_tex(pal_df, f"Per-palette accuracy of the Full Model on the TEST split only ({SEED_TXT}). "
                      "All palettes are present in training, so this measures consistency across palettes, not "
                      "generalisation to an unseen palette.", "tab:cross_palette", "table_cross_palette.tex")
    print("\nTABLE D — cross-palette, TEST split only (Full Model)\n" + pal_df.to_string(index=False))
    SUMMARY += ["## D · Cross-palette accuracy, TEST split only (Full Model)\n", md_table(pal_df), ""]

# ── key numbers in words ────────────────────────────────────────────────────
facts = []
if "Full Model" in by:
    facts.append(f"Full Model (Table V 'Ours'): emotion {fmt(vals('Full Model', g('emo.acc')))} %, "
                 f"identity {fmt(vals('Full Model', g('id.acc')))} %.")
for n in ORDER_ABL:
    if n not in (BASELINE, "Full Model"):
        facts.append(f"{n}: Δemotion vs baseline {dlt(n, BASELINE, g('emo.acc'))} pp; identity {fmt(vals(n, g('id.acc')))} %.")
if "Full Model + SimCLR init (N7)" in by:
    facts.append(f"SimCLR (N7) vs Full Model: Δemotion {dlt('Full Model + SimCLR init (N7)', 'Full Model', g('emo.acc'))} pp, "
                 f"Δidentity {dlt('Full Model + SimCLR init (N7)', 'Full Model', g('id.acc'))} pp.")
if "Full Model + CLUB" in by:
    mi0 = [r["hist"][0]["mi"] for r in by["Full Model + CLUB"].values()]
    mi1 = [r["hist"][-1]["mi"] for r in by["Full Model + CLUB"].values()]
    facts.append(f"CLUB: MI estimate first epoch {fmt(mi0, 1, 2)} → last epoch {fmt(mi1, 1, 2)}; "
                 f"Δemotion vs Full Model {dlt('Full Model + CLUB', 'Full Model', g('emo.acc'))} pp.")
for n in ORDER_LOSS[1:]:
    facts.append(f"{LOSS_LABEL.get(n, n)}: identity {fmt(vals(n, g('id.acc')))} % "
                 f"(Δ vs CE {dlt(n, 'Full Model', g('id.acc'))} pp).")
print("\nKEY NUMBERS\n" + "\n".join("  • " + f for f in facts))
SUMMARY += ["## Key numbers\n"] + ["- " + f for f in facts] + [""]

## 11 · Figures — ablation, training curves, CLUB, task-specific Grad-CAM

Colours: blue = emotion / train, orange = identity / validation. Grad-CAM uses one blue ramp (light → dark = low → high attention) over a grayscale copy of the image, so the attention map is not confused with the thermal palette.

In [ ]:
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e4e3df"
BLUE, ORANGE = "#2a78d6", "#eb6834"
SEQ = LinearSegmentedColormap.from_list("seq_blue", ["#cde2fb", "#6da7ec", "#2a78d6", "#184f95", "#0d366b"])
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "font.size": 9, "axes.titlesize": 10,
                     "axes.titlelocation": "left", "axes.edgecolor": MUTED, "axes.labelcolor": INK,
                     "xtick.color": MUTED, "ytick.color": MUTED, "text.color": INK,
                     "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True, "legend.frameon": False})

def savefig(fig, name):
    path = os.path.join(OUT_DIR, name)
    fig.savefig(path, bbox_inches="tight", facecolor="white"); plt.show(); plt.close(fig)
    print(f"  ✓ {name}")

# ── ablation dot plot (two panels, one per task — never a dual axis) ────────
if ORDER_ABL:
    names = ORDER_ABL; y = np.arange(len(names))[::-1]
    fig, axes = plt.subplots(1, 2, figsize=(10, 0.38 * len(names) + 1.3), sharey=True)
    for ax, key, title in ((axes[0], "emo.acc", "Emotion test accuracy (%)"), (axes[1], "id.acc", "Identity test accuracy (%)")):
        m = [clean(vals(n, g(key)), 100).mean() for n in names]
        s = [clean(vals(n, g(key)), 100).std(ddof=1) if len(clean(vals(n, g(key)), 100)) > 1 else 0 for n in names]
        if BASELINE in names:
            ax.axvline(m[names.index(BASELINE)], color=MUTED, lw=1, ls="--")
        ax.errorbar(m, y, xerr=s, fmt="o", color=BLUE, ms=6, lw=1.5, capsize=3)
        ax.set_title(title); ax.grid(axis="y", visible=False)
    axes[0].set_yticks(y); axes[0].set_yticklabels(names)
    fig.text(0.01, -0.02, f"Mean ± std over {N_SEEDS} seed(s); dashed line = baseline. Split: {SPLIT_MODE}, {EP_TXT}.",
             color=MUTED, fontsize=8)
    fig.tight_layout(); savefig(fig, "fig_ablation.png")

# ── Full Model training curves (first seed) ─────────────────────────────────
if "Full Model" in by:
    h = by["Full Model"][sorted(by["Full Model"])[0]]["hist"]; ep = [x["ep"] for x in h]
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
    mk = dict(marker="o", ms=3)
    axes[0].plot(ep, [x["loss"] for x in h], color=BLUE, lw=2, **mk); axes[0].set_title("Training loss")
    for ax, tr, va, t in ((axes[1], "tr_emo", "val_emo", "Emotion accuracy (%)"), (axes[2], "tr_id", "val_id", "Identity accuracy (%)")):
        ax.plot(ep, [x[tr] * 100 for x in h], color=BLUE, lw=2, label="Train", **mk)
        ax.plot(ep, [x[va] * 100 for x in h], color=ORANGE, lw=2, label="Validation", **mk)
        ax.set_title(t); ax.legend(loc="best")
    for ax in axes:
        ax.set_xlabel("Epoch")
        if len(ep) <= 10: ax.set_xticks(ep)
        else: ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    fig.tight_layout(); savefig(fig, "fig_full_model_curves.png")

# ── CLUB MI estimate per epoch ──────────────────────────────────────────────
if "Full Model + CLUB" in by:
    fig, ax = plt.subplots(figsize=(5.5, 3.2))
    curves = [[x["mi"] for x in r["hist"]] for r in by["Full Model + CLUB"].values()]
    for c_ in curves:
        ax.plot(range(1, len(c_) + 1), c_, color=GRID, lw=1)
    L = min(len(c_) for c_ in curves)
    ax.plot(range(1, L + 1), np.mean([c_[:L] for c_ in curves], 0), color=BLUE, lw=2, marker="o", ms=4)
    ax.set_title(r"CLUB estimate of MI($f_e$; $f_i$) per epoch"); ax.set_xlabel("Epoch"); ax.set_ylabel("MI estimate")
    if L <= 10: ax.set_xticks(range(1, L + 1))
    else: ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    fig.text(0.01, -0.04, "Blue = mean over seeds; grey = individual seeds.", color=MUTED, fontsize=8)
    fig.tight_layout(); savefig(fig, "fig_club_mi.png")

In [ ]:
class GradCAM:
    """Grad-CAM on the last EfficientNet-B0 block (7×7), one map per task head."""
    def __init__(self, model):
        self.model, self.act, self.grad = model, None, None
        self.hook = model.backbone[-1].register_forward_hook(self._save)
    def _save(self, module, inp, out):
        self.act = out
        if out.requires_grad:
            out.register_hook(lambda gr: setattr(self, "grad", gr))
    def __call__(self, x, pal, task):
        self.model.zero_grad(set_to_none=True)
        with torch.enable_grad():
            eo, io, _, _, _ = self.model(x, pal)
            logits = eo if task == "emo" else io
            c = int(logits.argmax(1))
            logits[0, c].backward()
        w = self.grad.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((w * self.act).sum(1))[0].detach().float().cpu().numpy()
        cam = cv2.resize(cam, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
        span = cam.max() - cam.min()
        return ((cam - cam.min()) / span if span > 1e-12 else np.zeros_like(cam)), c
    def close(self):
        self.hook.remove()

def overlay(img_rgb, cam, max_alpha=0.75):
    g_ = cv2.cvtColor(cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE)), cv2.COLOR_RGB2GRAY).astype(np.float32) / 255.0
    base = np.repeat(g_[..., None], 3, axis=2)
    a = (cam * max_alpha)[..., None]
    return np.clip(base * (1 - a) + SEQ(cam)[..., :3] * a, 0, 1)

if RUN_GRADCAM and os.path.exists(FULL_WEIGHTS):
    model = ThermalMTL(use_pain=True, id_loss="ce").to(DEVICE)
    model.load_state_dict(torch.load(FULL_WEIGHTS, map_location=DEVICE)); model.eval()
    engine, tf = GradCAM(model), val_tf()
    H = IMG_SIZE; THIRDS = [(0, H // 3), (H // 3, 2 * H // 3), (2 * H // 3, H)]
    REGIONS = ["Forehead (top third)", "Eyes / nose (middle third)", "Mouth / chin (bottom third)"]
    rng_ = random.Random(0); shares = {"emo": [], "id": []}; corr = []; examples = []
    for task, pool in (("emo", EMO_TE), ("id", ID_TE)):
        pick = pool[:]; rng_.shuffle(pick)
        for j, i in enumerate(pick[:GRADCAM_N]):
            s = SAMPLES[i]; img = load_rgb(s["path"])
            x = tf(image=img)["image"][None].to(DEVICE); p = torch.tensor([s["pal_idx"]], device=DEVICE)
            cam_e, pe = engine(x, p, "emo"); cam_i, pi_ = engine(x, p, "id")
            cam = cam_e if task == "emo" else cam_i
            tot_ = cam.sum() + 1e-8
            shares[task].append([cam[a:b].sum() / tot_ for a, b in THIRDS])
            if cam_e.std() > 0 and cam_i.std() > 0:
                corr.append(np.corrcoef(cam_e.ravel(), cam_i.ravel())[0, 1])
            if j < 3:
                examples.append((s, img, cam_e, pe, cam_i, pi_))
    engine.close()

    reg = pd.DataFrame({"Face region": REGIONS,
                        "Emotion head": [fmt(np.array(shares["emo"])[:, k], 100, 1) if shares["emo"] else "---" for k in range(3)],
                        "Identity head": [fmt(np.array(shares["id"])[:, k], 100, 1) if shares["id"] else "---" for k in range(3)]})
    corr_txt = f"{np.mean(corr):.2f} ± {np.std(corr, ddof=1):.2f}" if len(corr) > 1 else "n/a"
    corr_tex = corr_txt.replace("±", r"$\pm$")
    reg.to_csv(os.path.join(OUT_DIR, "table_gradcam_regions.csv"), index=False)
    write_tex(reg, "Share of Grad-CAM attention per face third (mean $\\pm$ std over test images; emotion head on "
                   f"emotion images, identity head on identity images). Regions are fixed image thirds, not detected "
                   f"landmarks. Mean correlation between the two heads' maps: {corr_tex}.",
              "tab:gradcam_regions", "table_gradcam_regions.tex")
    print("Grad-CAM attention share per face third (%)\n" + reg.to_string(index=False))
    print(f"Mean Pearson correlation between emotion-head and identity-head maps: {corr_txt}")
    SUMMARY += ["## E · Grad-CAM attention share per face third (%)\n", md_table(reg),
                f"\nMean correlation between the two heads' maps on the same image: {corr_txt}\n"]

    # example grid: input | emotion-head Grad-CAM | identity-head Grad-CAM
    rows = len(examples)
    fig, axes = plt.subplots(rows, 3, figsize=(7.2, 2.35 * rows))
    axes = np.atleast_2d(axes)
    for r, (s, img, cam_e, pe, cam_i, pi_) in enumerate(examples):
        truth = s["cls"] if s["task"] == "emo" else f"Person {s['cls']}"
        panels = [(cv2.resize(img, (IMG_SIZE, IMG_SIZE)), f"Input · {truth} · {s['pal']}"),
                  (overlay(img, cam_e), f"Emotion head → {EMOTIONS[pe]}"),
                  (overlay(img, cam_i), f"Identity head → Person {PERSONS[pi_]}")]
        for c_, (im_, title) in enumerate(panels):
            axes[r, c_].imshow(im_); axes[r, c_].set_title(title, fontsize=8); axes[r, c_].axis("off")
    fig.colorbar(ScalarMappable(norm=Normalize(0, 1), cmap=SEQ), ax=axes.ravel().tolist(), fraction=0.025, pad=0.02,
                 label="Grad-CAM (normalised)")
    savefig(fig, "fig_gradcam_task_specific.png")

    # attention share per region: two series → legend + direct labels
    fig, ax = plt.subplots(figsize=(6, 3.2)); xs = np.arange(3); w = 0.38
    for off, task, col, lab in ((-w / 2 - 0.01, "emo", BLUE, "Emotion head (emotion images)"),
                                (w / 2 + 0.01, "id", ORANGE, "Identity head (identity images)")):
        if shares[task]:
            mvals = np.array(shares[task]).mean(0) * 100
            ax.bar(xs + off, mvals, w, color=col, label=lab)
            for xx, vv in zip(xs + off, mvals):
                ax.text(xx, vv + 0.8, f"{vv:.0f}%", ha="center", fontsize=8, color=INK)
    ax.set_xticks(xs); ax.set_xticklabels(REGIONS); ax.set_ylabel("Share of attention (%)")
    ax.set_title("Where each head looks"); ax.legend(loc="upper left"); ax.grid(axis="x", visible=False)
    fig.tight_layout(); savefig(fig, "fig_gradcam_regions.png")
    del model
elif RUN_GRADCAM:
    print("Grad-CAM skipped: the Full Model weights for the first seed were not found "
          f"({FULL_WEIGHTS}). They are saved when that run finishes.")

## 12 · Save `summary.md` and zip everything for download

In [ ]:
SUMMARY += ["## Notes for the paper\n",
            "- All runs use one fixed split (`split_" + SPLIT_MODE + ".json`), images loaded in colour, AdamW lr 3e-4, "
            f"weight decay {WEIGHT_DECAY:g}, batch {BATCH_SIZE} (8+8), {EPOCHS} epochs, best epoch chosen on validation.",
            "- Augmentation actually used: HorizontalFlip 0.5; Affine (±5% shift, 0.9–1.1 scale, ±10°) 0.5; "
            "GaussNoise std 0.01–0.05 (×255 ≈ 2.6–12.8) 0.35; GaussianBlur kernel 3–5 0.2; brightness/contrast ±0.3 0.4; "
            "CLAHE clip 2.0 0.2; CoarseDropout 1–3 holes 0.2.",
            "- N2 (orthogonality) is applied to every image; ArcFace/CosFace predict with their own cosine classifier.",
            "- PaIN uses the image's palette label at test time (it must be known at inference).",
            "- SimCLR (N7) is pre-trained on TRAIN images only."]
open(os.path.join(OUT_DIR, "summary.md"), "w").write("\n".join(SUMMARY) + "\n")

flat = []
for r in runs:
    row = {"config": r["config"], "seed": r["seed"], "train_min": r["train_min"]}
    for t in ("emo", "id"):
        for m_ in ("acc", "prec", "rec", "f1"):
            row[f"{t}_{m_}"] = r[t][m_]
    for k_, v_ in r.get("probes", {}).items():
        row[f"probe_{k_}"] = v_
    flat.append(row)
if flat:
    pd.DataFrame(flat).to_csv(os.path.join(OUT_DIR, "all_runs.csv"), index=False)

zip_path = shutil.make_archive(OUT_DIR, "zip", OUT_DIR)
print(f"✓ summary.md, CSV/TEX tables and figures are in {OUT_DIR}")
print(f"✓ everything zipped → {zip_path}")
print("\nWhere each output goes in the paper:")
for a, b in [("Table VI (ablation)", "table_ablation.tex · fig_ablation.png"),
             ("Table V 'Ours' row", "Full Model row of table_ablation.tex"),
             ("Table IV (cross-palette)", "table_cross_palette.tex (test split only)"),
             ("Table VII (identity losses)", "table_losses.tex"),
             ("New: mechanism evidence", "table_mechanisms.tex"),
             ("Fig. 5 (Grad-CAM)", "fig_gradcam_task_specific.png · table_gradcam_regions.tex"),
             ("CLUB figure (same split)", "fig_club_mi.png")]:
    print(f"  {a:<30} ← {b}")
print(f"\nLeakage verdict for this split: {LEAK_VERDICT}")